#  Lab 11: Basic NLP Concepts & Text Preprocessing

---

##  Aim
To understand fundamental Natural Language Processing (NLP) concepts and apply a complete text preprocessing pipeline including tokenization, stopword removal, stemming, lemmatization, and POS tagging.

##  Theory

### What is NLP?
**Natural Language Processing (NLP)** is a branch of AI that enables computers to understand, interpret, and generate human language. Raw text is messy — it contains noise, inconsistencies, and linguistic variation. Before any ML model can learn from text, the data must be **preprocessed and cleaned**.

### The NLP Preprocessing Pipeline

```
Raw Text
   │
   ▼
Lowercasing  →  Remove Punctuation  →  Tokenization
   │
   ▼
Remove Stopwords  →  Stemming / Lemmatization
   │
   ▼
POS Tagging  →  Named Entity Recognition
   │
   ▼
Clean, Structured Text Ready for ML
```

### Key Concepts
| Concept | Description | Example |
|---|---|---|
| **Tokenization** | Splitting text into words or sentences | `"I love NLP"` → `['I', 'love', 'NLP']` |
| **Stopwords** | Common words with little meaning | `the, is, in, at, which` |
| **Stemming** | Chop word endings using rules (fast, rough) | `running` → `run`, `studies` → `studi` |
| **Lemmatization** | Reduce to dictionary base form (slower, accurate) | `running` → `run`, `studies` → `study` |
| **POS Tagging** | Assign grammatical roles to each word | `NLP(NN) is(VBZ) fun(JJ)` |
| **NER** | Identify named entities in text | `Google(ORG)`, `India(GPE)` |

---

##  Part 1: Setup & Installation

In [1]:
# Install required libraries
!pip install nltk spacy -q
!python -m spacy download en_core_web_sm -q

✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [2]:
import nltk
import spacy
import re
import string
from collections import Counter

# Download all required NLTK data
nltk_resources = [
    'punkt', 'punkt_tab', 'stopwords',
    'averaged_perceptron_tagger', 'averaged_perceptron_tagger_eng',
    'wordnet', 'omw-1.4'
]
for r in nltk_resources:
    nltk.download(r, quiet=True)

from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, SnowballStemmer
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag, ne_chunk

# Load spaCy model
nlp = spacy.load("en_core_web_sm")

print("✅ All libraries and resources loaded!")

✅ All libraries and resources loaded!


In [3]:
# Our working corpus — a realistic paragraph for all experiments
sample_text = """
Natural Language Processing (NLP) is a fascinating field of Artificial Intelligence.
It enables machines to read, understand, and derive meaning from human language.
Companies like Google, Amazon, and OpenAI are investing heavily in NLP research.
In 2024, large language models transformed how developers build intelligent applications.
The models are trained on billions of words and can perform tasks like translation,
summarization, and question-answering with remarkable accuracy!!!
"""

print("📄 Working corpus:")
print(sample_text)

📄 Working corpus:

Natural Language Processing (NLP) is a fascinating field of Artificial Intelligence.
It enables machines to read, understand, and derive meaning from human language.
Companies like Google, Amazon, and OpenAI are investing heavily in NLP research.
In 2024, large language models transformed how developers build intelligent applications.
The models are trained on billions of words and can perform tasks like translation,
summarization, and question-answering with remarkable accuracy!!!



---
##  Part 2: Text Cleaning

Text from real sources (social media, PDFs, websites) contains noise: special characters, extra spaces, HTML tags, numbers, etc. **Cleaning is always the first step.**

In [4]:
def clean_text(text):
    """
    Full text cleaning pipeline:
    1. Convert to lowercase
    2. Remove URLs
    3. Remove HTML tags
    4. Remove punctuation
    5. Remove extra whitespace
    """
    # Step 1: Lowercase
    text = text.lower()
    
    # Step 2: Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    
    # Step 3: Remove HTML tags
    text = re.sub(r'<.*?>', '', text)
    
    # Step 4: Remove punctuation and special characters
    text = re.sub(r'[^\w\s]', '', text)
    
    # Step 5: Remove digits
    text = re.sub(r'\d+', '', text)
    
    # Step 6: Collapse multiple spaces
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text


# Apply cleaning
cleaned_text = clean_text(sample_text)

print("🔴 Before Cleaning:")
print(sample_text[:200])
print("\n🟢 After Cleaning:")
print(cleaned_text[:200])

🔴 Before Cleaning:

Natural Language Processing (NLP) is a fascinating field of Artificial Intelligence.
It enables machines to read, understand, and derive meaning from human language.
Companies like Google, Amazon, an

🟢 After Cleaning:
natural language processing nlp is a fascinating field of artificial intelligence it enables machines to read understand and derive meaning from human language companies like google amazon and openai 


In [5]:
# Demonstrate on a noisy real-world example
noisy_examples = [
    "Check this out!!! Visit https://example.com for MORE info :) #AI #NLP",
    "<p>The model scored <b>95%</b> accuracy on the test set.</p>",
    "   WHY   is   Python   the   BEST   language???   "
]

print(f"{'Original':<55} → Cleaned")
print("-" * 100)
for ex in noisy_examples:
    cleaned = clean_text(ex)
    print(f"{ex[:52]:<55} → {cleaned}")

Original                                                → Cleaned
----------------------------------------------------------------------------------------------------
Check this out!!! Visit https://example.com for MORE    → check this out visit for more info ai nlp
<p>The model scored <b>95%</b> accuracy on the test     → the model scored accuracy on the test set
   WHY   is   Python   the   BEST   language???         → why is python the best language


---
##  Part 3: Tokenization

**Tokenization** splits text into smaller units called *tokens*. Tokens can be words, sentences, or even characters. It is the **gateway step** — every downstream NLP task depends on it.

In [6]:
# --- Word Tokenization ---
# NLTK's word_tokenize handles contractions, punctuation, abbreviations intelligently
word_tokens = word_tokenize(cleaned_text)

print(f"📌 Word Tokenization")
print(f"Total tokens : {len(word_tokens)}")
print(f"First 15     : {word_tokens[:15]}")

📌 Word Tokenization
Total tokens : 65
First 15     : ['natural', 'language', 'processing', 'nlp', 'is', 'a', 'fascinating', 'field', 'of', 'artificial', 'intelligence', 'it', 'enables', 'machines', 'to']


In [7]:
# --- Sentence Tokenization ---
# Splits paragraph into individual sentences (useful for summarization, translation)
sent_tokens = sent_tokenize(sample_text.strip())

print(f"📌 Sentence Tokenization")
print(f"Total sentences: {len(sent_tokens)}\n")
for i, sent in enumerate(sent_tokens, 1):
    print(f"  [{i}] {sent.strip()}")

📌 Sentence Tokenization
Total sentences: 6

  [1] Natural Language Processing (NLP) is a fascinating field of Artificial Intelligence.
  [2] It enables machines to read, understand, and derive meaning from human language.
  [3] Companies like Google, Amazon, and OpenAI are investing heavily in NLP research.
  [4] In 2024, large language models transformed how developers build intelligent applications.
  [5] The models are trained on billions of words and can perform tasks like translation,
summarization, and question-answering with remarkable accuracy!!
  [6] !


In [8]:
# --- Comparing Tokenization Methods ---
test = "It's a well-known fact that Dr. Smith won't stop teaching NLP!"

# Method 1: Naive split (bad)
naive = test.split()

# Method 2: NLTK word_tokenize (good)
nltk_tok = word_tokenize(test)

# Method 3: spaCy tokenizer (best)
spacy_tok = [token.text for token in nlp(test)]

print(f"Original  : {test}")
print(f"\n❌ Naive split  ({len(naive):2d} tokens): {naive}")
print(f"✅ NLTK         ({len(nltk_tok):2d} tokens): {nltk_tok}")
print(f"✅ spaCy        ({len(spacy_tok):2d} tokens): {spacy_tok}")
print("\n📌 NLTK and spaCy correctly split contractions like \"won't\" → \"wo\", \"n't\"")

Original  : It's a well-known fact that Dr. Smith won't stop teaching NLP!

❌ Naive split  (11 tokens): ["It's", 'a', 'well-known', 'fact', 'that', 'Dr.', 'Smith', "won't", 'stop', 'teaching', 'NLP!']
✅ NLTK         (14 tokens): ['It', "'s", 'a', 'well-known', 'fact', 'that', 'Dr.', 'Smith', 'wo', "n't", 'stop', 'teaching', 'NLP', '!']
✅ spaCy        (16 tokens): ['It', "'s", 'a', 'well', '-', 'known', 'fact', 'that', 'Dr.', 'Smith', 'wo', "n't", 'stop', 'teaching', 'NLP', '!']

📌 NLTK and spaCy correctly split contractions like "won't" → "wo", "n't"


---
##  Part 4: Stopword Removal

**Stopwords** are high-frequency words that add grammatical structure but little semantic value (`the, is, a, in, on, at`).  
Removing them reduces vocabulary size and improves signal-to-noise ratio for ML models.

In [9]:
stop_words = set(stopwords.words('english'))

print(f"Total English stopwords in NLTK: {len(stop_words)}")
print(f"\nSample stopwords:")
print(sorted(list(stop_words))[:30])

Total English stopwords in NLTK: 198

Sample stopwords:
['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't"]


In [10]:
# Apply stopword removal on our cleaned tokens
filtered_tokens = [w for w in word_tokens if w.lower() not in stop_words]

print(f"Before stopword removal : {len(word_tokens)} tokens")
print(f"After stopword removal  : {len(filtered_tokens)} tokens")
print(f"Reduction               : {len(word_tokens) - len(filtered_tokens)} tokens removed ({((len(word_tokens)-len(filtered_tokens))/len(word_tokens)*100):.1f}%)")
print(f"\n✅ Remaining tokens:")
print(filtered_tokens)

Before stopword removal : 65 tokens
After stopword removal  : 45 tokens
Reduction               : 20 tokens removed (30.8%)

✅ Remaining tokens:
['natural', 'language', 'processing', 'nlp', 'fascinating', 'field', 'artificial', 'intelligence', 'enables', 'machines', 'read', 'understand', 'derive', 'meaning', 'human', 'language', 'companies', 'like', 'google', 'amazon', 'openai', 'investing', 'heavily', 'nlp', 'research', 'large', 'language', 'models', 'transformed', 'developers', 'build', 'intelligent', 'applications', 'models', 'trained', 'billions', 'words', 'perform', 'tasks', 'like', 'translation', 'summarization', 'questionanswering', 'remarkable', 'accuracy']


In [11]:
# Custom stopwords — domain-specific additions
# In ML projects, you often add domain words that carry no task-specific meaning
custom_stops = stop_words.union({'also', 'however', 'therefore', 'thus', 'hence'})

custom_filtered = [w for w in word_tokens if w.lower() not in custom_stops]
print(f"With custom stopwords: {len(custom_filtered)} tokens")
print("📌 Always customize stopwords for your specific domain!")

With custom stopwords: 45 tokens
📌 Always customize stopwords for your specific domain!


---
##  Part 5: Stemming

**Stemming** aggressively chops word suffixes using rule-based heuristics to arrive at a common *stem*. It is **fast** but sometimes produces non-words (over-stemming).

Common stemmers: **Porter** (classic), **Snowball** (improved, multilingual)

In [12]:
porter   = PorterStemmer()
snowball = SnowballStemmer("english")

# Words to stem
test_words = [
    'running', 'runs', 'runner', 'ran',
    'studies', 'studying', 'studied',
    'happily', 'happiness', 'happy',
    'organization', 'organizing', 'organized',
    'better', 'best', 'good'
]

print(f"{'Word':<15} | {'Porter Stem':<15} | {'Snowball Stem'}")
print("-" * 50)
for word in test_words:
    p = porter.stem(word)
    s = snowball.stem(word)
    flag = "⚠️" if p != s else ""
    print(f"{word:<15} | {p:<15} | {s}  {flag}")

print("\n📌 ⚠️ marks cases where Porter and Snowball disagree")

Word            | Porter Stem     | Snowball Stem
--------------------------------------------------
running         | run             | run  
runs            | run             | run  
runner          | runner          | runner  
ran             | ran             | ran  
studies         | studi           | studi  
studying        | studi           | studi  
studied         | studi           | studi  
happily         | happili         | happili  
happiness       | happi           | happi  
happy           | happi           | happi  
organization    | organ           | organ  
organizing      | organ           | organ  
organized       | organ           | organ  
better          | better          | better  
best            | best            | best  
good            | good            | good  

📌 ⚠️ marks cases where Porter and Snowball disagree


In [13]:
# Apply Porter stemming to our filtered tokens
stemmed_tokens = [porter.stem(token) for token in filtered_tokens]

print("📌 Stemmed Token List:")
print(stemmed_tokens)

# Show over-stemming problem
print("\n⚠️  Over-stemming examples (stemmer creates non-words):")
for orig, stem in zip(filtered_tokens, stemmed_tokens):
    if orig.lower() != stem and len(stem) < len(orig) - 3:
        print(f"   '{orig}' → '{stem}'")

📌 Stemmed Token List:
['natur', 'languag', 'process', 'nlp', 'fascin', 'field', 'artifici', 'intellig', 'enabl', 'machin', 'read', 'understand', 'deriv', 'mean', 'human', 'languag', 'compani', 'like', 'googl', 'amazon', 'openai', 'invest', 'heavili', 'nlp', 'research', 'larg', 'languag', 'model', 'transform', 'develop', 'build', 'intellig', 'applic', 'model', 'train', 'billion', 'word', 'perform', 'task', 'like', 'translat', 'summar', 'questionansw', 'remark', 'accuraci']

⚠️  Over-stemming examples (stemmer creates non-words):
   'fascinating' → 'fascin'
   'intelligence' → 'intellig'
   'applications' → 'applic'
   'summarization' → 'summar'
   'questionanswering' → 'questionansw'
   'remarkable' → 'remark'


---
##  Part 6: Lemmatization

**Lemmatization** converts a word to its **dictionary base form (lemma)** using vocabulary and morphological analysis.  
It is **slower** than stemming but always produces valid words.

> 💡 Rule of thumb: Use **stemming** for search engines (speed matters). Use **lemmatization** for NLP tasks where correctness matters (sentiment analysis, text classification).

In [14]:
lemmatizer = WordNetLemmatizer()

# Lemmatizer needs POS context for best results
# 'v' = verb, 'n' = noun, 'a' = adjective, 'r' = adverb
test_words = [
    ('running',      'v'),
    ('studies',      'v'),
    ('happily',      'r'),
    ('better',       'a'),
    ('organizations','n'),
    ('was',          'v'),
    ('geese',        'n'),
    ('mice',         'n'),
    ('went',         'v'),
]

print(f"{'Word':<16} | {'POS':<6} | {'Stem (Porter)':<16} | {'Lemma'}")
print("-" * 60)
for word, pos in test_words:
    stem  = porter.stem(word)
    lemma = lemmatizer.lemmatize(word, pos=pos)
    print(f"{word:<16} | {pos:<6} | {stem:<16} | {lemma}")

print("\n📌 Notice: 'went'→'go', 'geese'→'goose' — only lemmatization handles these correctly!")

Word             | POS    | Stem (Porter)    | Lemma
------------------------------------------------------------
running          | v      | run              | run
studies          | v      | studi            | study
happily          | r      | happili          | happily
better           | a      | better           | good
organizations    | n      | organ            | organization
was              | v      | wa               | be
geese            | n      | gees             | goose
mice             | n      | mice             | mouse
went             | v      | went             | go

📌 Notice: 'went'→'go', 'geese'→'goose' — only lemmatization handles these correctly!


In [15]:
# spaCy lemmatization (automatic POS detection — more practical)
doc = nlp(" ".join(filtered_tokens))

lemmatized_tokens = [token.lemma_ for token in doc if not token.is_space]

print("📌 spaCy Lemmatized Tokens (with automatic POS):")
print(lemmatized_tokens)

# Side-by-side comparison
print("\n📊 Comparison: Original → Stemmed → Lemmatized")
print("-" * 60)
for orig, stem, lemma in zip(filtered_tokens[:10], stemmed_tokens[:10], lemmatized_tokens[:10]):
    print(f"{orig:<18} → {stem:<18} → {lemma}")

📌 spaCy Lemmatized Tokens (with automatic POS):
['natural', 'language', 'processing', 'nlp', 'fascinating', 'field', 'artificial', 'intelligence', 'enable', 'machine', 'read', 'understand', 'derive', 'meaning', 'human', 'language', 'company', 'like', 'google', 'amazon', 'openai', 'invest', 'heavily', 'nlp', 'research', 'large', 'language', 'model', 'transform', 'developer', 'build', 'intelligent', 'application', 'model', 'train', 'billion', 'word', 'perform', 'task', 'like', 'translation', 'summarization', 'questionanswere', 'remarkable', 'accuracy']

📊 Comparison: Original → Stemmed → Lemmatized
------------------------------------------------------------
natural            → natur              → natural
language           → languag            → language
processing         → process            → processing
nlp                → nlp                → nlp
fascinating        → fascin             → fascinating
field              → field              → field
artificial         → artifici    

---
##  Part 7: Part-of-Speech (POS) Tagging

**POS Tagging** assigns a grammatical label to each word — Noun, Verb, Adjective, etc.  
This is essential for information extraction, parsing, and named entity recognition.

### Common POS Tags (Penn Treebank)
| Tag | Meaning | Example |
|---|---|---|
| NN | Noun, singular | `model` |
| NNS | Noun, plural | `models` |
| VBZ | Verb, 3rd person singular | `is` |
| VBG | Verb, gerund | `running` |
| JJ | Adjective | `intelligent` |
| RB | Adverb | `quickly` |
| IN | Preposition | `in`, `on`, `at` |
| DT | Determiner | `the`, `a` |

In [16]:
# POS tagging with NLTK
pos_sentence = "The intelligent machine learning model accurately predicts stock prices."
pos_tokens   = word_tokenize(pos_sentence)
pos_tags     = pos_tag(pos_tokens)

print("📌 NLTK POS Tagging:")
print(f"\nSentence: {pos_sentence}\n")
print(f"{'Token':<20} | POS Tag")
print("-" * 35)
for token, tag in pos_tags:
    print(f"{token:<20} | {tag}")

📌 NLTK POS Tagging:

Sentence: The intelligent machine learning model accurately predicts stock prices.

Token                | POS Tag
-----------------------------------
The                  | DT
intelligent          | JJ
machine              | NN
learning             | VBG
model                | NN
accurately           | RB
predicts             | VBZ
stock                | NN
prices               | NNS
.                    | .


In [17]:
# spaCy POS tagging — richer tags + dependency info
doc2 = nlp(pos_sentence)

print("📌 spaCy POS Tagging (with explanation):")
print(f"\n{'Token':<20} | {'POS':<8} | {'Tag':<8} | {'Dep':<12} | Explanation")
print("-" * 75)
for token in doc2:
    print(f"{token.text:<20} | {token.pos_:<8} | {token.tag_:<8} | {token.dep_:<12} | {spacy.explain(token.tag_)}")

📌 spaCy POS Tagging (with explanation):

Token                | POS      | Tag      | Dep          | Explanation
---------------------------------------------------------------------------
The                  | DET      | DT       | det          | determiner
intelligent          | ADJ      | JJ       | amod         | adjective (English), other noun-modifier (Chinese)
machine              | NOUN     | NN       | compound     | noun, singular or mass
learning             | NOUN     | NN       | compound     | noun, singular or mass
model                | NOUN     | NN       | nsubj        | noun, singular or mass
accurately           | ADV      | RB       | advmod       | adverb
predicts             | VERB     | VBZ      | ROOT         | verb, 3rd person singular present
stock                | NOUN     | NN       | compound     | noun, singular or mass
prices               | NOUN     | NNS      | dobj         | noun, plural
.                    | PUNCT    | .        | punct        | pun

In [18]:
# Extract only NOUNS and VERBS from our full corpus
doc3 = nlp(sample_text)

nouns = [token.text for token in doc3 if token.pos_ == "NOUN"]
verbs = [token.lemma_ for token in doc3 if token.pos_ == "VERB"]
adjs  = [token.text for token in doc3 if token.pos_ == "ADJ"]

print(f"📌 Extracted from corpus:")
print(f"\nNouns ({len(nouns)}): {list(set(nouns))}")
print(f"\nVerbs ({len(verbs)}): {list(set(verbs))}")
print(f"\nAdjectives ({len(adjs)}): {list(set(adjs))}")

📌 Extracted from corpus:

Nouns (18): ['language', 'accuracy', 'machines', 'models', 'translation', 'Companies', 'meaning', 'field', 'developers', 'question', 'tasks', 'billions', 'summarization', 'research', 'words', 'applications']

Verbs (10): ['understand', 'train', 'perform', 'invest', 'transform', 'build', 'answer', 'derive', 'enable', 'read']

Adjectives (5): ['large', 'fascinating', 'remarkable', 'human', 'intelligent']


---
##  Part 8: Named Entity Recognition (NER)

**NER** identifies and classifies proper nouns — people, organizations, locations, dates — in text.  
It is widely used in information extraction, knowledge graphs, and search engines.

In [19]:
# NER with spaCy
ner_text = """
Elon Musk founded SpaceX in 2002 in Hawthorne, California.
Google, headquartered in Mountain View, was acquired by Alphabet Inc. in 2015.
The World Health Organization declared COVID-19 a pandemic in March 2020.
India's Prime Minister visited Washington D.C. last Monday for trade discussions.
"""

doc_ner = nlp(ner_text)

print(f"{'Entity':<25} | {'Label':<12} | Description")
print("-" * 65)
for ent in doc_ner.ents:
    print(f"{ent.text:<25} | {ent.label_:<12} | {spacy.explain(ent.label_)}")

Entity                    | Label        | Description
-----------------------------------------------------------------
Elon Musk                 | PERSON       | People, including fictional
2002                      | DATE         | Absolute or relative dates or periods
Hawthorne                 | GPE          | Countries, cities, states
California                | GPE          | Countries, cities, states
Google                    | ORG          | Companies, agencies, institutions, etc.
Mountain View             | GPE          | Countries, cities, states
Alphabet Inc.             | ORG          | Companies, agencies, institutions, etc.
2015                      | DATE         | Absolute or relative dates or periods
The World Health Organization | ORG          | Companies, agencies, institutions, etc.
March 2020                | DATE         | Absolute or relative dates or periods
India                     | GPE          | Countries, cities, states
Washington D.C.           | GPE     

In [20]:
# Group entities by type
from collections import defaultdict

entity_groups = defaultdict(list)
for ent in doc_ner.ents:
    entity_groups[ent.label_].append(ent.text)

print("📌 Entities grouped by type:")
for label, entities in sorted(entity_groups.items()):
    print(f"  {label:<12}: {entities}")

📌 Entities grouped by type:
  DATE        : ['2002', '2015', 'March 2020', 'last Monday']
  GPE         : ['Hawthorne', 'California', 'Mountain View', 'India', 'Washington D.C.']
  ORG         : ['Google', 'Alphabet Inc.', 'The World Health Organization']
  PERSON      : ['Elon Musk']


---
##  Part 9: Word Frequency Analysis

After preprocessing, analyzing word frequency helps understand **what a document is about**. This forms the basis of keyword extraction and topic modeling.

In [21]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')  # non-interactive backend

# Larger corpus for meaningful frequency analysis
large_text = """
Machine learning is a method of data analysis that automates analytical model building.
It is based on the idea that systems can learn from data, identify patterns and make decisions.
Machine learning algorithms are trained using data. The training data is used to learn patterns.
Deep learning is a subset of machine learning. Deep learning models use neural networks.
Neural networks are inspired by the human brain. The brain processes data through neurons.
Data science combines machine learning, statistics, and domain expertise to extract insights.
Insights from data help organizations make better data-driven decisions in the real world.
"""

# Full preprocessing pipeline
cleaned    = clean_text(large_text)
tokens     = word_tokenize(cleaned)
no_stops   = [w for w in tokens if w not in stop_words and len(w) > 2]
lemmatized = [WordNetLemmatizer().lemmatize(w) for w in no_stops]

# Frequency count
freq = Counter(lemmatized)
top_words = freq.most_common(15)

words, counts = zip(*top_words)

# Plot
plt.figure(figsize=(12, 5))
bars = plt.bar(words, counts, color=plt.cm.Blues_r([i/15 for i in range(15)]))
plt.title("Top 15 Most Frequent Words (after preprocessing)", fontsize=14, fontweight='bold')
plt.xlabel("Words")
plt.ylabel("Frequency")
plt.xticks(rotation=45, ha='right')
for bar, count in zip(bars, counts):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
             str(count), ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.savefig('word_frequency.png', dpi=120, bbox_inches='tight')
plt.show()
print("\n📊 Top 15 words:", dict(top_words))


📊 Top 15 words: {'data': 7, 'learning': 6, 'machine': 4, 'model': 2, 'learn': 2, 'pattern': 2, 'make': 2, 'decision': 2, 'deep': 2, 'neural': 2, 'network': 2, 'brain': 2, 'insight': 2, 'method': 1, 'analysis': 1}


/var/folders/mq/zx6g7lz93_92x5gk9r5_9g2h0000gn/T/ipykernel_30466/296844654.py:40: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
##  Part 10: Complete Preprocessing Pipeline (All Steps Together)

Putting everything together into a **reusable function** — exactly what you'd use in a real ML project.

In [22]:
def full_nlp_pipeline(text, use_lemma=True, verbose=True):
    """
    Complete NLP preprocessing pipeline:
    clean → tokenize → remove stopwords → stem/lemmatize
    
    Parameters:
        text      : raw input string
        use_lemma : if True uses lemmatization, else stemming
        verbose   : if True prints each stage
    Returns:
        list of preprocessed tokens
    """
    steps = {}

    # Stage 1: Clean
    steps['1. Raw']    = text.strip()
    cleaned            = clean_text(text)
    steps['2. Cleaned'] = cleaned

    # Stage 2: Tokenize
    tokens             = word_tokenize(cleaned)
    steps['3. Tokens'] = tokens

    # Stage 3: Remove stopwords + short tokens
    stop_words_set     = set(stopwords.words('english'))
    filtered           = [w for w in tokens if w not in stop_words_set and len(w) > 2]
    steps['4. No Stops'] = filtered

    # Stage 4: Normalize (lemmatize or stem)
    if use_lemma:
        lem    = WordNetLemmatizer()
        final  = [lem.lemmatize(w) for w in filtered]
        method = 'Lemmatized'
    else:
        ps     = PorterStemmer()
        final  = [ps.stem(w) for w in filtered]
        method = 'Stemmed'
    steps[f'5. {method}'] = final

    if verbose:
        for stage, result in steps.items():
            if isinstance(result, list):
                print(f"\n{'='*55}")
                print(f"📌 Stage {stage}")
                print(f"   Count: {len(result)} | Preview: {result[:8]}")
            else:
                print(f"\n{'='*55}")
                print(f"📌 Stage {stage}")
                print(f"   {result[:120]}..." if len(result) > 120 else f"   {result}")

    return final


# Run on our sample text
test_input = """
The researchers are rapidly developing new machine learning algorithms.
These algorithms have been applied to medical imaging, natural language processing,
and autonomous vehicle systems worldwide!!!
"""

result = full_nlp_pipeline(test_input, use_lemma=True)
print(f"\n\n✅ Final Output ({len(result)} tokens):")
print(result)


📌 Stage 1. Raw
   The researchers are rapidly developing new machine learning algorithms.
These algorithms have been applied to medical im...

📌 Stage 2. Cleaned
   the researchers are rapidly developing new machine learning algorithms these algorithms have been applied to medical ima...

📌 Stage 3. Tokens
   Count: 25 | Preview: ['the', 'researchers', 'are', 'rapidly', 'developing', 'new', 'machine', 'learning']

📌 Stage 4. No Stops
   Count: 18 | Preview: ['researchers', 'rapidly', 'developing', 'new', 'machine', 'learning', 'algorithms', 'algorithms']

📌 Stage 5. Lemmatized
   Count: 18 | Preview: ['researcher', 'rapidly', 'developing', 'new', 'machine', 'learning', 'algorithm', 'algorithm']


✅ Final Output (18 tokens):
['researcher', 'rapidly', 'developing', 'new', 'machine', 'learning', 'algorithm', 'algorithm', 'applied', 'medical', 'imaging', 'natural', 'language', 'processing', 'autonomous', 'vehicle', 'system', 'worldwide']


In [23]:
# --- Compare pipeline with lemma vs stem ---
print("📊 Lemmatization vs Stemming on same input:")
print("-" * 50)

with_lemma = full_nlp_pipeline(test_input, use_lemma=True,  verbose=False)
with_stem  = full_nlp_pipeline(test_input, use_lemma=False, verbose=False)

print(f"{'Lemmatized':<22} | {'Stemmed'}")
print("-" * 50)
for l, s in zip(with_lemma, with_stem):
    flag = "⚠️" if l != s else ""
    print(f"{l:<22} | {s}  {flag}")

📊 Lemmatization vs Stemming on same input:
--------------------------------------------------
Lemmatized             | Stemmed
--------------------------------------------------
researcher             | research  ⚠️
rapidly                | rapidli  ⚠️
developing             | develop  ⚠️
new                    | new  
machine                | machin  ⚠️
learning               | learn  ⚠️
algorithm              | algorithm  
algorithm              | algorithm  
applied                | appli  ⚠️
medical                | medic  ⚠️
imaging                | imag  ⚠️
natural                | natur  ⚠️
language               | languag  ⚠️
processing             | process  ⚠️
autonomous             | autonom  ⚠️
vehicle                | vehicl  ⚠️
system                 | system  
worldwide              | worldwid  ⚠️
